#read table from silver layer

In [0]:
df = spark.read.format("delta").table("`ttc-lakehouse`.silver.routes")

##select relevent attributes for building the gold route dimension 

In [0]:
df = df.select('route_id','agency_id','route_short_name','route_long_name','route_type')

##Engorce correct data type for GTFS route_type code 

In [0]:
df = df.withColumn('route_type',df.route_type.cast('int'))

## Enrich route dimension with descriptive transport mode 

In [0]:
from pyspark.sql.functions import when

df = df.withColumn(
    "route_type_desc",
    when(df.route_type == 0, "tram")
    .when(df.route_type == 1, "subway")
    .when(df.route_type == 2, "rail")
    .when(df.route_type == 3, "bus")
    .when(df.route_type == 4, "ferry")
    .when(df.route_type == 5, "cable tram")
    .when(df.route_type == 6, "gondola")
    .when(df.route_type == 7, "funicular")
    .otherwise(None)
)

##write into Gold layer

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("`ttc-lakehouse`.gold.dim_routes")